In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split

In [2]:
import io
import zipfile
import urllib.request

# Tatoeba English-German sentence pairs (CC BY 2.0 FR) from manythings.org
request = urllib.request.Request("https://www.manythings.org/anki/deu-eng.zip", headers={"User-Agent": "ml-notebooks"})
with urllib.request.urlopen(request) as response:
    archive = zipfile.ZipFile(io.BytesIO(response.read()))
dataset = pd.read_csv(archive.open("deu.txt"), sep="\t", header=None, usecols=[0, 1], names=["en", "de"], quoting=3).head(64000)
dataset = dataset.dropna()
dataset.head()

,en,de
0,Go.,Geh.
1,Hi.,Hallo!
2,Hi.,Grüß Gott!
3,Run!,Lauf!
4,Run.,Lauf!


In [3]:
VOCABULARY_SIZE = 5000
MAX_SEQUENCE_LENGTH = 32
UNITS = 256 
EMBEDDING_DIM = 256
BUFFER = 5000
BATCH_SIZE = 32

english_vectorizer = tf.keras.layers.TextVectorization(
    standardize = "lower_and_strip_punctuation",
    max_tokens = MAX_SEQUENCE_LENGTH, 
    output_mode = 'int',
    output_sequence_length = MAX_SEQUENCE_LENGTH
)

german_vectorizer = tf.keras.layers.TextVectorization(
    standardize = "lower_and_strip_punctuation",
    max_tokens = MAX_SEQUENCE_LENGTH, 
    output_mode = 'int',
    output_sequence_length = MAX_SEQUENCE_LENGTH
)

In [4]:
def standardize_text(text, is_target):
    w = text.lower()
    w = re.sub(r'[" "]+', " ",  w)
    w = w.strip()

    if is_target:
        w = 'starttoken ' + w + ' endtoken'
    return w

dataset["eng"] = dataset["en"].apply(lambda x: standardize_text(x, is_target=False))
dataset["ger"] = dataset["de"].apply(lambda x: standardize_text(x, is_target=True))

english_lang = dataset["eng"].tolist()
german_lang = dataset["ger"].tolist()

print(english_lang[:3])
print(german_lang[:3])

['go.', 'hi.', 'hi.']
['starttoken geh. endtoken', 'starttoken hallo! endtoken', 'starttoken grüß gott! endtoken']


In [5]:
english_train, english_val, german_train, german_val = train_test_split(english_lang, german_lang, test_size=0.2, random_state=42)

english_vectorizer.adapt(english_train)
german_vectorizer.adapt(german_train)

input_vector = english_vectorizer(english_train)
output_vector = english_vectorizer(german_train)

In [6]:
input_vocab = {index: word for index, word in enumerate(english_vectorizer.get_vocabulary())}
output_vocab = {index: word for index, word in enumerate(german_vectorizer.get_vocabulary())}

print(dict(list(input_vocab.items())[:30]))
print(dict(list(output_vocab.items())[:30]))

{0: '', 1: '[UNK]', 2: np.str_('tom'), 3: np.str_('i'), 4: np.str_('you'), 5: np.str_('is'), 6: np.str_('a'), 7: np.str_('it'), 8: np.str_('the'), 9: np.str_('to'), 10: np.str_('im'), 11: np.str_('do'), 12: np.str_('me'), 13: np.str_('was'), 14: np.str_('that'), 15: np.str_('are'), 16: np.str_('this'), 17: np.str_('he'), 18: np.str_('we'), 19: np.str_('my'), 20: np.str_('did'), 21: np.str_('have'), 22: np.str_('dont'), 23: np.str_('your'), 24: np.str_('its'), 25: np.str_('can'), 26: np.str_('like'), 27: np.str_('not'), 28: np.str_('youre'), 29: np.str_('were')}
{0: '', 1: '[UNK]', 2: np.str_('starttoken'), 3: np.str_('endtoken'), 4: np.str_('ich'), 5: np.str_('tom'), 6: np.str_('ist'), 7: np.str_('sie'), 8: np.str_('das'), 9: np.str_('nicht'), 10: np.str_('es'), 11: np.str_('du'), 12: np.str_('wir'), 13: np.str_('hat'), 14: np.str_('er'), 15: np.str_('ein'), 16: np.str_('habe'), 17: np.str_('bin'), 18: np.str_('zu'), 19: np.str_('sind'), 20: np.str_('die'), 21: np.str_('war'), 22: np.s

In [7]:
def vectorize(input_text, target_text):
    encoder_input = english_vectorizer(input_text)

    decoder_input_text = [text.replace(" endtoken",  "") for text in target_text]
    decoder_output_text = [text.replace("starttoken ", "") for text in target_text]

    decoder_input = german_vectorizer(decoder_input_text)
    decoder_output = german_vectorizer(decoder_output_text)

    return {"input_1": encoder_input, "input_2": decoder_input}, decoder_output

In [8]:
x_train, y_train = vectorize(english_train, german_train)
x_val, y_val = vectorize(english_val, german_val)

train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_dataset = train_dataset.shuffle(BUFFER).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val))
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [9]:
class Encoder(tf.keras.Model):
    def __init__(self, vocabulary_size, embedding_dim, units):
        super(Encoder, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocabulary_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(units, return_sequences=True, return_state=True)

    def call(self, x):
        x = self.embedding(x)
        output, state_h, state_c = self.lstm(x)
        return output, state_h

In [10]:
# Bahdanau Attention
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, prev_decoder_state, encoder_state):
        query_with_time_axis = tf.expand_dims(prev_decoder_state, 1)
        scores = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(encoder_state)))
        attention_weights = tf.nn.softmax(scores, axis=1)
        context_vector = attention_weights * encoder_state
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

In [11]:
class Decoder(tf.keras.Model):
    def __init__(self, vocabulary_size, embeeding_dim, units, sequence_length):
        super(Decoder, self).__init__()
        self.emebdding =  tf.keras.layers.Embedding(vocabulary_size, embeeding_dim)
        self.lstm = tf.keras.layers.LSTM(units, return_state=True)
        self.sequence_length = sequence_length
        self.attention = BahdanauAttention(units)
        self.func = tf.keras.layers.Dense(vocabulary_size, activation='softmax')

    def call(self, x, hidden, shifted_target):
        outputs = []
        shifted_target = self.emebdding(shifted_target)
        for t in range (0, self.sequence_length):
            context_vector, _ = self.attention(hidden, x)
            dec_input = context_vector + shifted_target[:, t]
            output, hidden_h, hidden_c = self.lstm(tf.expand_dims(dec_input, 1))
            outputs.append(output)
        outputs = tf.stack(outputs)
        outputs = tf.transpose(outputs, perm=[1, 0, 2])
        outputs = self.func(outputs)
        return outputs

In [12]:
enc_input = tf.keras.Input(shape=(MAX_SEQUENCE_LENGTH,), name="input_1", dtype='int64')
dec_input = tf.keras.Input(shape=(MAX_SEQUENCE_LENGTH,), name="input_2", dtype='int64')

encoder_model = Encoder(VOCABULARY_SIZE, EMBEDDING_DIM, UNITS)
output, state_h = encoder_model(enc_input)

decoder_model = Decoder(VOCABULARY_SIZE, EMBEDDING_DIM, UNITS, MAX_SEQUENCE_LENGTH)
outputs = decoder_model(output, state_h, dec_input)

In [13]:
model = tf.keras.Model([enc_input, dec_input], outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_1             │ (None, 32)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder (Encoder)   │ [(None, 32, 256), │  1,805,312 │ input_1[0][0]     │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_2             │ (None, 32)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder (Decoder)   │ (None, 32, 5000)  │  3,222,153 │ encoder[0][0],    │
│                     │                   │            │ encoder[0][1],    │
│                     │                   │            │ input_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,027,465 (19.18 MB)

 Trainable params: 5,027,465 (19.18 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
history = model.fit(train_dataset, validation_data=val_dataset, epochs=1)

C:\Users\jorda\Code\ml-notebooks\courses\deep-learning\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


   1/1600 ━━━━━━━━━━━━━━━━━━━━ 9:55:28 22s/step - accuracy: 0.0000e+00 - loss: 8.5192

   2/1600 ━━━━━━━━━━━━━━━━━━━━ 2:25 91ms/step - accuracy: 0.4204 - loss: 8.5012      

   3/1600 ━━━━━━━━━━━━━━━━━━━━ 2:26 92ms/step - accuracy: 0.5632 - loss: 8.4771

   4/1600 ━━━━━━━━━━━━━━━━━━━━ 2:25 91ms/step - accuracy: 0.6323 - loss: 8.4383

   5/1600 ━━━━━━━━━━━━━━━━━━━━ 2:24 90ms/step - accuracy: 0.6746 - loss: 8.3708

... (1,592 outputs left out)

1598/1600 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.9087 - loss: 0.3811

1599/1600 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.9087 - loss: 0.3810

1600/1600 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.9087 - loss: 0.3810

1600/1600 ━━━━━━━━━━━━━━━━━━━━ 224s 126ms/step - accuracy: 0.9087 - loss: 0.3810 - val_accuracy: 0.9277 - val_loss: 0.2371


In [15]:
def translate(sentence):
    tokenized = english_vectorizer([sentence])
    shifted = 'starttoken'
    results = ''

    for i in range(MAX_SEQUENCE_LENGTH):
        shift_tokenized = german_vectorizer([shifted])
        output = model.predict([tokenized, shift_tokenized])

        word_index = tf.argmax(output, axis=-1)[0][i].numpy()

        current_word = german_vectorizer.get_vocabulary()[word_index]
        if current_word == 'endtoken':
            break
        shifted += " " + current_word
        if i != 0:
            results += " "
        results += current_word
    return results

In [16]:
translate("good morning")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11s/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step


'[UNK]'

In [17]:
translate("he loves you")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step


... (55 outputs left out)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step


'er [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK]'